In [15]:
import time
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
import yfinance as yf

In [16]:
st.set_page_config(page_title="Live Stock Dashboard", layout="wide")
st.title("ð Live Stock Market Stream")

2026-11-26 10:28:39.218 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-26 10:28:39.220 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-26 10:28:39.221 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-26 10:28:39.222 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [17]:
with st.sidebar:
    st.header("Settings")
    # Defaulting to Reliance Industries (NSE) or user-selected ticker
    ticker = st.text_input("Enter Stock Ticker", value="RELIANCE.NS").upper()
    refresh_rate = st.slider("Refresh Interval (seconds)", min_value=5, max_value=60, 
value=10)
    chart_type = st.radio("Chart Type", ["Candlestick", "Line"])
    st.caption("Tip: Use `.NS` for NSE (e.g., `TCS.NS`) or standard tickers for US stocks (e.g., `AAPL`).")

2026-11-26 10:28:39.651 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-26 10:28:39.652 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-26 10:28:39.653 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-26 10:28:39.653 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-26 10:28:39.654 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-26 10:28:39.656 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-26 10:28:39.657 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-26 10:28:39.658 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [40]:
def fetch_intraday_data(symbol: str) -> pd.DataFrame:
    stock = yf.Ticker(symbol)
    df = stock.history(period="1d", interval="1m")
    return df

In [43]:
@st.fragment(run_every=refresh_rate)
def render_live_dashboard(symbol: str, c_type: str):
    df = fetch_intraday_data(symbol)
    if df.empty:
        st.warning(f"No live data available for **{symbol}**. The market may be closed or the ticker is invalid.")
        return

    # Move your cell 42 code inside the function (indented):
    latest_price = df["Close"].iloc[-1]
    prev_close = df["Open"].iloc[0]
    price_diff = latest_price - prev_close
    pct_change = (price_diff / prev_close) * 100
    day_high = df["High"].max()
    day_low = df["Low"].min()
    total_vol = df["Volume"].sum()
    
    # Display metrics with st.metric(), etc.

In [41]:

@st.fragment(run_every=refresh_rate)
def render_live_dashboard(symbol: str, c_type: str):
    df = fetch_intraday_data(symbol)
    if df.empty:
        st.warning(f"No live data available for **{symbol}**. The market may be closed or the ticker is invalid.")
        return

In [44]:
latest_price=df["Close"].iloc[-1]
prev_close=df["Open"].iloc[0]
price_diff=latest_price - prev_close
pct_change=(price_diff / prev_close) * 100
day_high=df["High"].max()
day_low=df["Low"].min()
total_vol=df["Volume"].sum()

NameError: name 'df' is not defined

In [ ]:

    # Display KPI Metrics
    col1, col2, col3, col4 = st.columns(4)
    col1.metric(
        label=f"{symbol} Current Price",
        value=f"{latest_price:,.2f}",
        delta=f"{price_diff:+.2f} ({pct_change:+.2f}%)"
    )
    col2.metric(label="Day High", value=f"{day_high:,.2f}")
    col3.metric(label="Day Low", value=f"{day_low:,.2f}")
    col4.metric(label="Intraday Volume", value=f"{total_vol:,}")
    # Build Interactive Plotly Chart
    fig = go.Figure()
    if c_type == "Candlestick":
        fig.add_trace(
            go.Candlestick(
                x=df.index,
                open=df["Open"],
                high=df["High"],
                low=df["Low"],
                close=df["Close"],
                name="Price"
            )
        )
    else:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["Close"],
                mode="lines",
                name="Close Price",
                line=dict(color="#00CC96", width=2)
            )
        )
    fig.update_layout(
        title=f"{symbol} Intraday Movement (1m Interval)",
        xaxis_title="Time",
        yaxis_title="Price",
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        height=500,
        margin=dict(l=20, r=20, t=40, b=20)
    )
    st.plotly_chart(fig, use_container_width=True)
    st.caption(f"Last updated: {datetime.now().strftime('%H:%M:%S')} | Auto
refreshing every {refresh_rate}s")
# Run the live component
render_live_dashboard(ticker, chart_type